# Extract POIs from pbf

This notebook loads pbf files of single European countries via pyrosm, then uses city boundary geojson files of all European cities (above 100k population or capitals) to extract and export a list of POIs (rail stations and schools) as gpkg files from the corresponding country.

## Preliminary step

All .osm.pbf country files need to be downloaded from [geofabrik](https://download.geofabrik.de/europe.html), renamed to just `countryname.osm.pbf`, and placed in the countries folder. The following country names need special renames to the following:
- north-macedonia.osm.pbf
- bosnia-and-herzegovina.osm.pbf
- ireland.osm.pbf

## Load packages

In [ ]:
import pyrosm
import csv
from growbikenet.functions import slugify
from growbikenet import constants
import os
from IPython.display import clear_output
import geopandas as gpd

## Parameters

In [ ]:
cities_path = "../cities/"
cityfilename = "cities.csv"
output_folder = "cityexport/"

## Load cities

In [16]:
with open(cities_path+'meta/'+cityfilename, mode='r') as infile:
    reader = csv.reader(infile, delimiter=";")
    header = next(reader)
    cities = {slugify(rows[0])+"_"+slugify(rows[3]): {header[0]: rows[0], header[1]: rows[1], header[2]: rows[2], header[3]: rows[3]} for rows in reader}

## Run

In [ ]:
cities_log = []          
steps_logs = []
cities_wout_pbf = []

def render():
    clear_output(wait=True)
    for line in cities_log:
        print(line)
    for step in steps_logs:
        print("    " + step)


passuntil = None # Set to a city name to ignore all previous cities. Set None to not ignore any cities. 
for cityid, city_info in cities.items():
    if city_info["name_en"] == passuntil:
        passuntil = None
    if passuntil is None:
        cities_log.append(city_info["name_en"]+", "+city_info["country_en"])
        steps_logs = []
        render()

        # Load city boundary
        try:
            boundary = gpd.read_file(cities_path+output_folder+'boundaries/'+cityid+".geojson")
        except:
            boundary = gpd.read_file(cities_path+output_folder+'boundaries/'+cityid+".shp")
        steps_logs.append("Loaded city boundary")
        render()
    
        # Load city pbf
        citypbf_file = cities_path+output_folder+"pbfs/"+cityid+".osm.pbf"
        if not os.path.exists(citypbf_file): 
            cities_wout_pbf.append(city_info["name_en"]+", "+city_info["country_en"])
            continue
        else:
            osm = pyrosm.OSM(citypbf_file)
            steps_logs.append("Loaded city .pbf file")
        render()

    
        # =======================================
        # Extract rail stations from pyrosm's city osm object, truncate to boundary, and save as city gpkg rail station
        citygpkg_file_rail_stations = cities_path+output_folder+"rail_stations/"+cityid+".gpkg"
        if not os.path.exists(citygpkg_file_rail_stations): # Create and save
            rail_stations = osm.get_pois(
                custom_filter = constants._PRESET_TAGS['rail']
            )
            rail_stations
            if rail_stations is not None:
                rail_stations = gpd.clip(rail_stations, boundary)
                if not len(rail_stations.is_empty) == 0:
                    rail_stations['geometry'].to_file(citygpkg_file_rail_stations)
                    steps_logs.append("Extracted rail stations")
                else:
                    steps_logs.append("No rail stations")
            else: 
                steps_logs.append("No rail stations")
            
        render()
            
    
        # =======================================
        # Extract schools from pyrosm's city osm object, truncate to boundary, and save as city gpkg schools
        citygpkg_file_schools = cities_path+output_folder+"schools/"+cityid+".gpkg"
        if not os.path.exists(citygpkg_file_schools): # Create and save
            schools = osm.get_pois(
                custom_filter = constants._PRESET_TAGS['school']
            )
            if schools is not None:
                schools = gpd.clip(schools, boundary)
                if not len(schools.is_empty) == 0:
                    schools['geometry'].to_file(citygpkg_file_schools)
                    steps_logs.append("Extracted schools")
                else:
                    steps_logs.append("No schools")
            else:
                steps_logs.append("No schools")
        render()